# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding #1 — "The Anatomy of Growing Content" (page 6, CONFIRMED)

**Claim:** Growing pages (impressions trending up) are on average 37.6% longer and 20% younger than declining pages. The paper frames this as a direct portfolio comparison, not a model.

**My methodology question:** This is a strong observational comparison, but it's still a comparison, not a controlled experiment — so the natural question is about *direction of causality and selection*. Do longer pages grow because they're longer, or do pages that are already growing get selected for expansion (a team notices early traction and invests more words into it)? The paper aggregates across 57 brands — if a few large or content-heavy brands dominate the "growing" bucket, the word-count gap could be a brand-mix effect rather than a universal content pattern. I'd ask: was this checked per-brand, or only at the pooled portfolio level? That's the same instinct behind grouping my own split by `client_id` — pooled numbers can hide the fact that a few big accounts are driving the whole pattern.

---

### Finding — ML Appendix, "What Predicts Growth?" (page 29, exploratory)

**Claim:** Logistic regression reaches 71% holdout accuracy predicting growing vs. declining pages, with content age as the strongest negative signal and recent impressions/days-visible as strongest positive signals.

**My methodology question:** This is the one I can ask most concretely, because I hit the same trap myself in my Week-5 model. Two questions: (1) **Where does the label come from, and does any feature share its time window?** My own growth/decline label is computed from a 30-day-vs-previous-30-day impression comparison — if this model's "Impressions" feature is the *same* recent 30-day window used to build the label (rather than a strictly-prior window), that's the exact label-derived-feature leak I found and removed in my own notebook. The paper doesn't state whether the feature windows are pre-label or overlapping. (2) **Is the 71% holdout split grouped by brand, or a random row split?** With 57 brands in the portfolio, a random split could let the model partly memorize brand-specific quirks instead of learning a pattern that generalizes to a brand it's never seen — which is precisely why I switched from a random split to a client-grouped split in Section 2 below, and saw the score change.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

np.random.seed(42)

import os
import urllib.request

csv_path = 'data/raw/content_refresh_anonymized.csv'
if os.path.exists(csv_path):
    os.remove(csv_path)
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
url = "https://raw.githubusercontent.com/velchan15/MachineLearning-InternshipStarter-FlyRank/main/data/raw/content_refresh_anonymized.csv"
urllib.request.urlretrieve(url, csv_path)

df = pd.read_csv(csv_path)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df['avg_position'] = df['avg_position'].replace(0, np.nan)

flagged_cols = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'avg_position']
for col in flagged_cols:
    df[f'has_{col}'] = df[col].notna().astype(int)
has_flag_cols = [f'has_{c}' for c in flagged_cols]

# Same feature set as Week 5, WITHOUT the *_last_30d columns (leakage guard from w05, re-applied here)
numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
categorical_features = [
    'content_type', 'main_intent', 'provider_used', 'model_used',
    'competition_level', 'age_tier', 'freshness_tier',
    'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier',
]

X_cols = numeric_features + categorical_features + has_flag_cols
X = df[X_cols].copy()
y = df['is_declining_label'].copy()
groups = df['client_id'].copy()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('flag', 'passthrough', has_flag_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order][:k].mean()


In [2]:
# BEFORE: naive random row-level split (what a first attempt often looks like)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
lr_random = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))])
lr_random.fit(X_train_r, y_train_r)
proba_random = lr_random.predict_proba(X_test_r)[:, 1]
p10_random = precision_at_k(proba_random, y_test_r, 10)
p50_random = precision_at_k(proba_random, y_test_r, 50)

# AFTER: grouped split by client_id (my actual Week-5 choice)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, X_test_g = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
y_train_g, y_test_g = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)

lr_grouped = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))])
lr_grouped.fit(X_train_g, y_train_g)
proba_grouped = lr_grouped.predict_proba(X_test_g)[:, 1]
p10_grouped = precision_at_k(proba_grouped, y_test_g, 10)
p50_grouped = precision_at_k(proba_grouped, y_test_g, 50)

client_overlap_random = len(set(groups.iloc[X_train_r.index]) & set(groups.iloc[X_test_r.index]))
client_overlap_grouped = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

comparison = pd.DataFrame([
    {'split': 'Random (ungrouped)', 'base_rate': round(y_test_r.mean(),3), 'precision_at_10': round(p10_random,3), 'precision_at_50': round(p50_random,3), 'client_overlap': client_overlap_random},
    {'split': 'Grouped by client_id', 'base_rate': round(y_test_g.mean(),3), 'precision_at_10': round(p10_grouped,3), 'precision_at_50': round(p50_grouped,3), 'client_overlap': client_overlap_grouped},
])
comparison


,split,base_rate,precision_at_10,precision_at_50,client_overlap
0,Random (ungrouped),0.542,0.9,0.94,31
1,Grouped by client_id,0.511,0.8,0.84,0


**Before/after result:**

| Split | Base rate | P@10 | P@50 | Client overlap train/test |
|---|---|---|---|---|
| Random (ungrouped) | 0.542 | 0.900 | 0.940 | 31 clients |
| Grouped by client_id | 0.511 | 0.800 | 0.840 | 0 clients |

The random split lets 31 of the same clients appear in both train and test, and the score is inflated as a result (P@50 0.94 vs 0.84). That 0.10 gap at P@50 is itself the finding: it's a rough estimate of how much of the random-split score was the model partly memorizing client-level quirks rather than learning something that transfers to a brand it's never seen. This mirrors exactly the methodology question I raised about the paper's own 71%-accuracy growth model in Section 1 — I can't verify their split design from the outside, but I can show concretely, on my own data, how large that gap can be.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Deliberately re-add the leaky feature found in Week 5 (impressions_last_30d) to confirm
# the harness is actually sensitive to leakage, not just quiet by default.
numeric_features_leaky = numeric_features + ['impressions_last_30d']
X_leaky = df[numeric_features_leaky + categorical_features + has_flag_cols].copy()

preprocessor_leaky = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features_leaky),
    ('flag', 'passthrough', has_flag_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])

X_train_leaky, X_test_leaky = X_leaky.iloc[train_idx].reset_index(drop=True), X_leaky.iloc[test_idx].reset_index(drop=True)
lr_leaky = Pipeline([('prep', preprocessor_leaky), ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))])
lr_leaky.fit(X_train_leaky, y_train_g)
proba_leaky = lr_leaky.predict_proba(X_test_leaky)[:, 1]
p10_leaky = precision_at_k(proba_leaky, y_test_g, 10)
p50_leaky = precision_at_k(proba_leaky, y_test_g, 50)

print(f"With impressions_last_30d added back : P@10={p10_leaky:.3f}  P@50={p50_leaky:.3f}")
print(f"Without it (my actual Week-5 model)  : P@10={p10_grouped:.3f}  P@50={p50_grouped:.3f}")


With impressions_last_30d added back : P@10=1.000  P@50=1.000
Without it (my actual Week-5 model)  : P@10=0.800  P@50=0.840


In [4]:
# Sanity-check the remaining (kept) numeric features for any other suspicious correlation with the label
corr_check = df[numeric_features + ['is_declining_label']].corr(numeric_only=True)['is_declining_label'] \
    .drop('is_declining_label').sort_values(key=abs, ascending=False)
corr_check.head(10)


,is_declining_label
days_with_impressions,0.190055
content_age_days,-0.163882
word_count,0.090157
days_since_last_update,0.081383
avg_position,-0.081304
char_count,0.072188
ctr,-0.061911
clicks_90d,-0.039680
engaged_sessions_90d,-0.035402
clicks_prev_30d,-0.028716


**Result:** re-adding `impressions_last_30d` pushes the score straight back to a perfect P@10 = 1.000, P@50 = 1.000 — confirming the test harness is genuinely sensitive to this leak, not just quietly ignoring it. Removing it drops the score back to the honest 0.800 / 0.840 from Section 2.

The remaining features top out at a correlation of 0.19 (`days_with_impressions`) with the label — nothing close to the near-1.0 pattern that flagged the original leak, so I don't have new evidence of a second hidden leak in the current feature set. `days_with_impressions` being the top legitimate signal makes sense descriptively (a page visible on fewer days recently is plausibly the one declining) rather than being suspiciously perfect.

**Attack checklist, applied:**
- [x] Timeline drawn: `*_prev_30d` window is fully pre-label; `*_last_30d` window excluded because it overlaps the label's own window
- [x] No label-derived or sibling columns in the features (confirmed by the deliberate re-add test above)
- [x] No product flags / existing FlyRank-style scores used as features
- [x] Split grouped by `client_id`
- [x] Base rate printed next to every metric (Section 2 table)
- [x] Top feature importance sanity-checked (Week-5 notebook; nothing suspiciously dominant)
- [x] Metrics computed out-of-fold on a held-out test split, never in-sample

# Sanity-check the remaining (kept) numeric features for any other suspicious correlation with the label
corr_check = df[numeric_features + ['is_declining_label']].corr(numeric_only=True)['is_declining_label'] \
    .drop('is_declining_label').sort_values(key=abs, ascending=False)
corr_check.head(10)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

| Original claim (Week 5) | Problem with the wording | Rewritten claim |
|---|---|---|
| "Logistic Regression wins outright — the simpler model is the better one here." | "Wins outright" reads as a universal, permanent conclusion rather than a result tied to one split, one sample, one metric. | On this client-grouped test split, Logistic Regression **measured** a higher precision@50 (0.84) than Random Forest (0.56) — an **observed**, sample-specific result, not a general claim that simpler models always outperform ensemble models. |
| "The model is over-weighting sheer traffic volume." | States the model's internal behavior as settled fact from a small (8-row) error sample. | Error review on the top-50 ranked test rows shows a **directional** pattern: false positives skew toward the highest-traffic tier, which is suggestive of over-reliance on volume-related features but is based on a small sample and would need a larger audit to confirm. |
| "Both models beat the baseline by a wide margin." | "Wide margin" is vague and reads as more authoritative than a single split's numbers support. | Both models scored higher precision@50 than the recomputed baseline rule on the same held-out test split (LR: 0.84, RF: 0.56, baseline: 0.32) — useful as **decision-support** for prioritizing which pages to review first, not as a guarantee of real-world performance at deployment. |

The common thread: last week's language stated results as settled, general facts. This week's versions tie every number back to *this* split, *this* sample, and *this* metric, and use the paper's own safe-claim vocabulary (observed, measured, directional, decision-support) instead of words like "wins outright" or "over-weighting" stated as fact.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.